In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 78 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 

In [2]:
%%writefile typecheck.l
%{
#include "typecheck.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%%

"int" {
    return INT;
}

"float" {
    return FLOAT;
}

[a-zA-Z_][a-zA-Z0-9_]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

"=" {
    return '=';
}

"+" {
    return '+';
}

"-" {
    return '-';
}

"*" {
    return '*';
}

"/" {
    return '/';
}

";" {
    return ';';
}

[ \t\n] {
    /* skip whitespace */
}

. {
    return yytext[0];
}

%%

int yywrap()
{
    return 1;
}

Writing typecheck.l


In [3]:
%%writefile typecheck.y
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

struct sym {
    char name[20];
    char type[10];
} table[50];

int n = 0;

void insert(char *name, char *type)
{
    strcpy(table[n].name, name);
    strcpy(table[n].type, type);
    n++;
}

char* typeOf(char *name)
{
    int i;

    for (i = 0; i < n; i++)
    {
        if (strcmp(table[i].name, name) == 0)
            return table[i].type;
    }

    return "undefined";
}

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM
%token INT FLOAT

%type <str> expr

%left '+' '-'
%left '*' '/'

%%

program:
    stmts
    ;

stmts:
    stmts stmt
    | stmt
    ;

stmt:
    decl
    | assign
    ;

decl:
    INT ID ';'
    {
        insert($2, "int");
        printf("Declared %s as int\n", $2);
    }

    | FLOAT ID ';'
    {
        insert($2, "float");
        printf("Declared %s as float\n", $2);
    }
    ;

assign:
    ID '=' expr ';'
    {
        char *lt = typeOf($1);

        if (strcmp(lt, "undefined") == 0)
            printf("Undefined variable: %s\n", $1);

        else if (strcmp(lt, $3) == 0)
            printf("No type mismatch in expression: %s = ...\n", $1);

        else
            printf("Type mismatch in assignment to %s\n", $1);
    }
    ;

expr:
    ID
    {
        char *t = typeOf($1);

        if (strcmp(t, "undefined") == 0)
            printf("Undefined variable: %s\n", $1);

        $$ = t;
    }

    | NUM
    {
        $$ = "int";
    }

    | expr '+' expr
    {
        $$ = (strcmp($1, $3) == 0) ? $1 : "mismatch";
    }

    | expr '-' expr
    {
        $$ = (strcmp($1, $3) == 0) ? $1 : "mismatch";
    }

    | expr '*' expr
    {
        $$ = (strcmp($1, $3) == 0) ? $1 : "mismatch";
    }

    | expr '/' expr
    {
        $$ = (strcmp($1, $3) == 0) ? $1 : "mismatch";
    }
    ;

%%

int main()
{
    printf("Enter declarations and expressions:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}

Writing typecheck.y


In [4]:
!rm -f typecheck.tab.c typecheck.tab.h lex.yy.c typecheck

In [5]:
!bison -d typecheck.y

In [6]:
!flex typecheck.l

In [7]:
!gcc lex.yy.c typecheck.tab.c -o typecheck -lfl

In [8]:
!printf "int a;\nint b;\na=b;\n" | ./typecheck

Enter declarations and expressions:
Declared a as int
Declared b as int
No type mismatch in expression: a = ...


In [9]:
!printf "float a;\nfloat b;\na=b;\n" | ./typecheck

Enter declarations and expressions:
Declared a as float
Declared b as float
No type mismatch in expression: a = ...


In [10]:
!printf "int a;\nint b;\nint c;\na=b+c;\n" | ./typecheck

Enter declarations and expressions:
Declared a as int
Declared b as int
Declared c as int
No type mismatch in expression: a = ...


In [11]:
!printf "int a;\na=b;\n" | ./typecheck

Enter declarations and expressions:
Declared a as int
Undefined variable: b
Type mismatch in assignment to a


In [12]:
!printf "int a;\nint b;\nint c;\na=b*c;\n" | ./typecheck

Enter declarations and expressions:
Declared a as int
Declared b as int
Declared c as int
No type mismatch in expression: a = ...
